In [12]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import pandas as pd

JupyterDash.infer_jupyter_proxy_config()

###########################
# Data Manipulation / Model
###########################
from CRUD_Python_Module import AnimalShelterCRUD

# FIX ME: update with your username and password
username = "aacuser"
password = "Biology1!"

# Connect to database via CRUD Module
db = AnimalShelterCRUD(username, password)

# Pull ALL documents for the initial (reset) view
df = pd.DataFrame.from_records(db.read({}))

# MongoDB returns _id as ObjectId which can crash the table; drop it if present
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Center(html.B(html.H1('CS-340 Dashboard'))),

    # Logo + unique identifier 
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '250px'}
        ),
        html.H4("Unique Identifier: Jean Karst"),
    ], style={'textAlign': 'center'}),

    html.Hr(),

    # Interactive filtering options
    html.Div([
        html.H3("Rescue Type Filter"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset (All)', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain / Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster / Individual Tracking', 'value': 'disaster'}
            ],
            value='reset',
            labelStyle={'display': 'block'}
        )
    ], style={'marginLeft': '20px'}),

    html.Hr(),

    # Interactive DataTable
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),

        # Usability features
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],

        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'minWidth': '100px', 'width': '150px', 'maxWidth': '300px'},
        style_header={'fontWeight': 'bold'}
    ),

    html.Br(),
    html.Hr(),

    # Side-by-side chart and map
    html.Div(className='row',
             style={'display': 'flex'},
             children=[
                 html.Div(
                     id='graph-id',
                     className='col s12 m6',
                     style={'flex': '1', 'paddingRight': '10px'}
                 ),
                 html.Div(
                     id='map-id',
                     className='col s12 m6',
                     style={'flex': '1', 'paddingLeft': '10px'}
                 )
             ])
])

#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    def breed_or_regex(breeds):
        # Build a Mongo $or set of regexes so "Mix" variants are included
        return {'$or': [{'breed': {'$regex': b, '$options': 'i'}} for b in breeds]}

    # Default query: reset/all
    query = {}

    if filter_type == 'water':
        # Water Rescue: Labrador Retriever Mix, Chesapeake Bay Retriever, Newfoundland
        # Intact Female, 26-156 weeks
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            **breed_or_regex(['Labrador Retriever', 'Chesapeake Bay Retriever', 'Newfoundland'])
        }

    elif filter_type == 'mountain':
        # Mountain/Wilderness: German Shepherd, Alaskan Malamute, Old English Sheepdog,
        # Siberian Husky, Rottweiler
        # Intact Male, 26-156 weeks
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            **breed_or_regex(['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler'])
        }

    elif filter_type == 'disaster':
        # Disaster/Tracking: Doberman Pinscher, German Shepherd, Golden Retriever,
        # Bloodhound, Rottweiler
        # Intact Male, 20-300 weeks
        query = {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
            **breed_or_regex(['Doberman Pinscher', 'German Shepherd', 'Golden Retriever', 'Bloodhound', 'Rottweiler'])
        }

    # Query Mongo via CRUD module
    dff = pd.DataFrame.from_records(db.read(query))

    # Drop _id if present
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Second chart (bar chart of top 10 breeds) based on current table view
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return [html.Div("No data to display.")]

    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' not in dff.columns:
        return [html.Div("Column 'breed' not found in data.")]

    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']
    breed_counts = breed_counts.head(10)

    fig = px.bar(
        breed_counts,
        x='breed',
        y='count',
        title='Top 10 Breeds in Current View'
    )
    fig.update_layout(xaxis_tickangle=-45)

    return [dcc.Graph(figure=fig)]


# Highlight selected columns
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Geo map updates from selected row in the table view
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [html.Div("No map data to display.")]

    dff = pd.DataFrame.from_dict(viewData)

    # Default to first row if nothing is selected
    row = 0 if not index else index[0]

    # Try to find lat/long columns (common names used in the dataset)
    # If your dataset uses different names, update these two lines.
    lat_col_candidates = ['location_lat', 'lat', 'latitude']
    lon_col_candidates = ['location_long', 'lon', 'longitude', 'lng']

    lat_col = next((c for c in lat_col_candidates if c in dff.columns), None)
    lon_col = next((c for c in lon_col_candidates if c in dff.columns), None)

    if lat_col is None or lon_col is None:
        return [html.Div("Latitude/Longitude columns not found for map.")]

    # Breed + Name columns (optional)
    breed_col = 'breed' if 'breed' in dff.columns else None
    name_col = 'name' if 'name' in dff.columns else None

    lat = dff.iloc[row][lat_col]
    lon = dff.iloc[row][lon_col]

    if pd.isna(lat) or pd.isna(lon):
        return [html.Div("Selected row has no latitude/longitude.")]

    tooltip_text = dff.iloc[row][breed_col] if breed_col else "Selected Animal"
    popup_name = dff.iloc[row][name_col] if name_col else "(no name)"

    # Austin TX center
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(str(tooltip_text)),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(str(popup_name))
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode
app.run_server()

Dash app running on https://buffalospend-catalogflame-3000.codio.io/proxy/8050/
